# Retraining by resource name with the raw API

This notebook shows every request required to resolve names to stable IDs, upload an immutable dataset version, and start retraining.

In [ ]:
# Scenario inputs
BUSINESS_CASE_NAME = "Estates Sell Prices"
DATASET_NAME = "sale-prices"
PIPELINE_NAME = "Estates Sell Prices - AutoFEML"
INPUT_FILE = None  # For example r"C:\data\sale-prices.csv"; None generates 30k demo records

## Connection and authentication

In automation, provide `ML_APP_ACCESS_TOKEN` through a secret manager. For local use, `getpass` prompts for a password without storing it in the notebook.

In [ ]:
import getpass
import os
from pathlib import Path

import requests

API = os.getenv("ML_APP_API_URL", "http://localhost:8000/api/v1").rstrip("/")
token = os.getenv("ML_APP_ACCESS_TOKEN", "").strip()
if not token:
    login_response = requests.post(
        f"{API}/auth/login",
        json={"login": input("Login: "), "password": getpass.getpass("Password: ")},
        timeout=30,
    )
    login_response.raise_for_status()
    token = login_response.json()["access_token"]
session = requests.Session()
session.headers["Authorization"] = f"Bearer {token}"

## Resolve names to IDs

Execution APIs use stable IDs. The following bounded metadata requests resolve each user-facing name.

In [ ]:
business_cases = session.get(f"{API}/business-cases", timeout=30).json()
business_case = next(item for item in business_cases if item["name"] == BUSINESS_CASE_NAME)
attachments = session.get(
    f"{API}/business-cases/{business_case['id']}/data-attachments", timeout=30
).json()
datasets = session.get(f"{API}/datasets", timeout=30).json()
attached_ids = {item["data_asset_id"] for item in attachments}
dataset = next(
    item for item in datasets
    if item["id"] in attached_ids and item["name"] == DATASET_NAME
)
pipelines = session.get(
    f"{API}/pipelines", params={"business_case_id": business_case["id"]}, timeout=30
).json()
pipeline = next(item for item in pipelines if item["name"] == PIPELINE_NAME)
versions = session.get(f"{API}/pipelines/{pipeline['id']}/versions", timeout=30).json()
pipeline_version = max(
    (item for item in versions if item["status"] == "published"),
    key=lambda item: item["version_number"],
)
print({"business_case_id": business_case["id"], "logical_dataset_id": dataset["logical_id"], "pipeline_id": pipeline["id"]})

## Prepare the input file

When `INPUT_FILE` is unset, the deterministic helper creates 10k base plus 20k synthetic records without making API requests.

In [ ]:
if INPUT_FILE:
    training_file = Path(INPUT_FILE)
else:
    import sys

    helpers = next(
        path for path in [Path.cwd(), Path.cwd() / "examples" / "API-usage"]
        if (path / "api_example_helpers.py").is_file()
    )
    sys.path.insert(0, str(helpers))
    from api_example_helpers import build_training_file

    training_file = build_training_file()
training_file

## Request 1: upload a new dataset version

Passing the resolved `logical_id` creates the next immutable version in the same dataset family.

In [ ]:
with training_file.open("rb") as file:
    response = session.post(
        f"{API}/datasets/upload",
        files={"file": (training_file.name, file, "text/csv")},
        data={"logical_id": dataset["logical_id"]},
        timeout=300,
    )
response.raise_for_status()
uploaded = response.json()
print(f"Uploaded {uploaded['name']} v{uploaded['version_number']} ({uploaded['row_count']:,} rows)")

## Request 2: start retraining

The request pins the latest published pipeline version. Its `latest` input policy resolves to the dataset version uploaded above.

In [ ]:
response = session.post(
    f"{API}/pipelines/{pipeline['id']}/runs",
    json={"pipeline_version_id": pipeline_version["id"]},
    timeout=120,
)
response.raise_for_status()
run = response.json()
print(f"Retraining started: run_id={run['id']}; status={run['status']}")